In [1]:
%pip install langchain-ollama langchain langchain-community pydantic>=2.0 --upgrade

zsh:1: 2.0 not found
Note: you may need to restart the kernel to use updated packages.


In [6]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

llm = ChatOllama(model="llama3.2", temperature=0.0)

In [10]:
# %pip install langchain
# %pip install langchain-community
%pip install docarray

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Modern LangChain imports
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from IPython.display import display, Markdown

file = "data.csv"
loader = CSVLoader(file_path=file)
documents = loader.load()


In [4]:
documents[0]

Document(metadata={'source': 'data.csv', 'row': 0}, page_content='product: Wireless Headphones\nreview: Amazing sound quality! Battery lasts all day and the noise cancellation works great. Highly recommend for music lovers.')

In [8]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
embed = embeddings.embed_query("Hi, I am trying to study Agentic AI")

In [13]:
print(len(embed))
print(embed[:5])

768
[-0.056607574, 0.03702436, -0.11754525, 0.024928043, 0.04979292]


In [12]:
db = DocArrayInMemorySearch.from_documents(documents, embeddings)

In [14]:
query = "Please suggest a gadget that has good reviews"
docs1 = db.similarity_search(query)

In [16]:
list(docs1)

[Document(metadata={'source': 'data.csv', 'row': 0}, page_content='product: Wireless Headphones\nreview: Amazing sound quality! Battery lasts all day and the noise cancellation works great. Highly recommend for music lovers.'),
 Document(metadata={'source': 'data.csv', 'row': 1}, page_content='product: Smart Watch\nreview: Tracks my fitness perfectly and syncs with my phone seamlessly. The heart rate monitor is accurate and the display is bright.'),
 Document(metadata={'source': 'data.csv', 'row': 10}, page_content='product: Desk Lamp\nreview: Adjustable brightness levels are perfect. USB charging port is a nice bonus. Modern design looks great on my desk.'),
 Document(metadata={'source': 'data.csv', 'row': 3}, page_content="product: Coffee Maker\nreview: Makes perfect coffee every morning. The programmable timer is convenient and it's easy to clean. Love it!")]

In [15]:
docs1[0]

Document(metadata={'source': 'data.csv', 'row': 0}, page_content='product: Wireless Headphones\nreview: Amazing sound quality! Battery lasts all day and the noise cancellation works great. Highly recommend for music lovers.')

In [18]:
qdocs1 = "".join(docs1[i].page_content for i in range(len(docs1)))

In [21]:
response = llm.invoke(f"{qdocs1} Question: Please list all the gadgets \
with a good review in a table in markdown and summarize each one.")

In [24]:
display(Markdown(response.content))

Here is the table of gadgets with good reviews:

| Product | Review Summary |
| --- | --- |
| Wireless Headphones | Excellent sound quality, long battery life, and effective noise cancellation. Ideal for music enthusiasts. |
| Smart Watch | Accurate fitness tracking, seamless phone syncing, and a bright display. Great for those who want to stay connected and on top of their health. |
| Desk Lamp | Adjustable brightness levels, convenient USB charging port, and a modern design that looks great on any desk. Perfect for task lighting or ambient illumination. |
| Coffee Maker | Consistently produces perfect coffee with ease, thanks to the programmable timer and easy cleaning process. A must-have for coffee lovers. |

Let me know if you'd like me to add anything else!

In [4]:
retriever = db.as_retriever()

NameError: name 'db' is not defined

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:
{context}

Question: {question}
"""
)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    |prompt
    | llm
    | StrOutputParser()
)

NameError: name 'retriever' is not defined

In [29]:
response = qa_chain.invoke("Please list all the gadgets with good reviews in a table \
in markdown and summarize each one.")

In [30]:
display(Markdown(response))

| Product | Review Summary |
| --- | --- |
| Desk Lamp | Adjustable brightness levels, USB charging port, modern design |
| Wireless Headphones | Amazing sound quality, long battery life, effective noise cancellation |
| Smart Watch | Tracks fitness perfectly, accurate heart rate monitor, bright display |
| Backpack | Durable material, multiple compartments, well-padded laptop sleeve |

Note: The summaries are based on the provided reviews and may not be exhaustive.

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown

# This block replaces: VectorstoreIndexCreator().from_loaders([loader])
llm = ChatOllama(model="llama3.2", temperature=0.0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")
db = DocArrayInMemorySearch.from_documents(
    CSVLoader("data.csv").load(), embeddings
)

# This block replaces: index.query("question", llm=llm)
qa_chain = (
    {
        "context": db.as_retriever() | (lambda docs: "\n\n".join(d.page_content for d in docs)),
        "question": RunnablePassthrough()
    }
    | ChatPromptTemplate.from_template("Context:\n{context}\n\nQuestion: {question}")
    | llm
    | StrOutputParser()
)

In [ ]:
display(Markdown(qa_chain.invoke("List all gadgets with good reviews in a markdown table and summarize each one.")))

In [31]:
examples = [
    {
        "query": "Does Smart Watch track heart rate?",
        "answer": "Yes"
    },
    {
        "query": "What product is good for allergies?",
        "answer": "Air Purifier"
    }
]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

qa_generation_prompt = ChatPromptTemplate.from_template(
    """Given the following document, generate a question and answer pair.
The question should beanswerable from the document.
Document: {doc}

Return the response in the json format with keys "question" and "answer".""")

qa_gen_chain = qa_generation_prompt | llm | StrOutputParser()

qa_pairs = []
for doc in documents:
    qa_pairs.append(qa_gen_chain.invoke({"doc": doc.page_content}))
display(qa_pairs)


['Here is the JSON response:\n\n{"question": "What are the key features of these wireless headphones?", "answer": "Amazing sound quality, Battery lasts all day, Noise cancellation works great."}',
 'Here\'s a JSON response:\n\n{"question": "Does the smart watch track fitness accurately?", "answer": "Yes, it tracks fitness perfectly."}',
 'Here is a JSON response with a question and answer pair based on the provided document:\n\n{"question": "What material is the laptop stand made of?", "answer": "Sturdy aluminum"}',
 'Here is a JSON response with a question and answer pair based on the document:\n\n{"question": "What feature of the coffee maker makes it convenient for users?", "answer": "The programmable timer"}',
 'Here\'s a JSON response:\n\n{"question": "What are the benefits of this yoga mat?", "answer": "Excellent grip, cushioning, non-slip surface, and suitable thickness for knees."}',
 'Here is a JSON response with a question and answer pair based on the document:\n\n{"question"